# Punto 5 — Comparación y elección de modelos SARIMA

En este punto se prueban distintos modelos para las cuatro series diarias de Rosario. Se busca elegir, para cada serie, el modelo que cometa menos errores al predecir datos que no utilizó para calcular sus parámetros.

## ¿Qué es un modelo SARIMA?

Un modelo **SARIMA** utiliza valores y errores anteriores de una serie para predecir sus próximos valores. También puede representar patrones que se repiten, como el ciclo semanal de los viajes en bicicleta.

Se escribe **SARIMA(p,d,q)(P,D,Q)ₛ**. Sus letras indican:

- **p:** cuántos valores anteriores se tienen en cuenta.
- **d:** cuántas veces se calcula la diferencia entre un valor y el anterior.
- **q:** cuántos errores anteriores se tienen en cuenta.
- **P, D y Q:** cumplen funciones similares, pero para el ciclo estacional.
- **s:** duración del ciclo. Por ejemplo, **s=7** representa una semana en datos diarios.

Un modelo sin parte estacional es un caso particular de esta familia y se denomina ARIMA. Si tampoco necesita diferencias, se denomina ARMA.

## ¿Cómo se elige el modelo?

Se separan los datos por fecha:

1. **Entrenamiento (Training Set): 2023–2024.** Se usa para calcular los parámetros y elegir el modelo.
2. **Prueba (Testing Set): 2025.** Se reserva para evaluar el modelo elegido en el punto 6.

Dentro del entrenamiento, se reservan los últimos **90 días** para comparar los modelos. Cada modelo se ajusta con los días anteriores y predice esos 90 días sin conocer sus valores reales. Este paso se llama **validación**.

Se elige el modelo con menor **MAE**, es decir, menor error absoluto promedio. Después se vuelven a calcular sus parámetros con todos los datos de 2023–2024.

Los puntos 2–4 ya analizaron la serie completa. Por eso, aunque los errores de 2025 no se usan para elegir el modelo, las decisiones iniciales pueden estar influidas por esa revisión previa.

## Modelos que se comparan

| Serie | Alternativas | Motivo |
|---|---|---|
| Viajes MBTB | 12 combinaciones, con y sin parte semanal | Evaluar el patrón de la semana y si conviene aplicar diferencias. |
| Temperatura y humedad | 4 modelos con los valores originales y otros 4 con diferencia anual | Comprobar si restar el valor de hace 365 días mejora la predicción. |
| Precipitación | 4 modelos con los valores originales | Las pruebas del punto 4 no mostraron una necesidad clara de aplicar diferencias. |

En temperatura y humedad, la **diferencia anual** consiste en restar a cada valor el de 365 días antes. Se modela esa diferencia y luego se vuelve a la unidad original para calcular el error. Estos modelos se escriben SARIMA(p,0,q)(0,1,0)₃₆₅. Las primeras 365 observaciones sirven como referencia y no entran en el ajuste de las diferencias. El período de 365 días es una aproximación, ya que el calendario incluye un año bisiesto.

Se prueba un conjunto limitado de opciones. No se incluyen términos que relacionen directamente los valores o errores de años anteriores, aparte de la diferencia anual. Para viajes tampoco se representan a la vez los ciclos semanal y anual. Por lo tanto, se elige **el mejor de los modelos probados**, no necesariamente el mejor de todos los posibles.

Los modelos sin diferencias incluyen un término constante. Los modelos con diferencias no incluyen un término adicional de crecimiento sostenido, llamado deriva.

## ¿Cómo se leen las tablas?

- **MAE de validación:** error absoluto promedio en los 90 días reservados. Es el criterio utilizado para elegir el modelo; cuanto menor, mejor.
- **AIC y BIC:** medidas que consideran tanto el ajuste como la cantidad de parámetros. Valores menores son preferibles cuando los modelos son comparables. No se usan para ordenar juntos modelos con distintas diferencias o distinta cantidad de datos utilizados.
- **Coeficiente:** valor calculado para cada parámetro del modelo.
- **Error estándar e intervalo de confianza:** indican la incertidumbre de esa estimación.
- **p-valor:** si es menor a 0,05, aporta evidencia de que el parámetro es distinto de cero, siempre que se cumplan los supuestos del modelo. Esto no asegura que el modelo prediga bien.

Si el cálculo de un modelo no logra converger, es decir, no alcanza una solución numérica estable, se descarta y se registra la advertencia.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/bicicletas_rosario.csv').exists())
if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))
from modelos_puntos56 import cargar, seleccionar, ajustar, ajustados, pronosticar, metricas, etiqueta
series = cargar(ROOT)
plt.rcParams.update({'figure.figsize': (12, 4), 'figure.dpi': 110})


In [2]:
OUT = ROOT / 'puntos/punto5'
OUT.mkdir(exist_ok=True)
seleccionados, comparaciones, parametros, resumen = {}, [], [], []
for nombre, s in series.items():
    train = s.loc[:'2024-12-31']
    c, r, tabla, avisos = seleccionar(train, nombre)
    seleccionados[nombre] = c
    tabla.insert(0, 'serie', nombre)
    comparaciones.append(tabla)
    ic = r.conf_int()
    par = pd.DataFrame({'coeficiente': r.params, 'error_estandar': r.bse,
                        'p_valor': r.pvalues, 'IC95_inferior': ic.iloc[:,0], 'IC95_superior': ic.iloc[:,1]})
    par.insert(0, 'serie', nombre)
    par.index.name = 'parametro'
    parametros.append(par.reset_index())
    elegida = tabla[tabla.seleccionado.eq(True)].iloc[0]
    resumen.append({'serie': nombre, 'modelo': etiqueta(c), 'n_train': len(train),
                    'n_estimacion': int(r.nobs), 'MAE_validacion': elegida.MAE_validacion,
                    'AIC_train': r.aic, 'BIC_train': r.bic, 'advertencias_train': avisos})
    display(Markdown(f'### {nombre}'))
    display(tabla.sort_values('MAE_validacion').round(3))
    display(Markdown(f'**Modelo seleccionado:** {etiqueta(c)}. Parámetros calculados con todos los datos de 2023–2024:'))
    display(par.round(4))
comparacion = pd.concat(comparaciones, ignore_index=True)
resumen = pd.DataFrame(resumen)
comparacion.to_csv(OUT / 'comparacion_sarima.csv', index=False)
pd.concat(parametros, ignore_index=True).to_csv(OUT / 'parametros_seleccionados.csv', index=False)
resumen.to_csv(OUT / 'modelos_seleccionados.csv', index=False)
(OUT / 'seleccion.json').write_text(json.dumps(seleccionados, ensure_ascii=False, indent=2))
display(resumen.round(3))


### Viajes MBTB

,serie,id,modelo,estado,familia,MAE_validacion,AIC,BIC,n_ajuste,advertencias,seleccionado
5,Viajes MBTB,5,"SARIMA(0, 1, 1)×(0, 1, 1, 7)",ok,serie original,424.572,9700.280,9713.631,641,,True
6,Viajes MBTB,6,"SARIMA(0, 1, 1)×(1, 0, 1, 7)",ok,serie original,425.190,9810.315,9828.160,641,,NaN
0,Viajes MBTB,0,"SARIMA(1, 0, 0)×(1, 0, 0, 7)",ok,serie original,430.885,9949.204,9967.056,641,,NaN
3,Viajes MBTB,3,"SARIMA(1, 0, 0)×(0, 0, 0, 0)",ok,serie original,433.569,10022.279,10035.668,641,,NaN
4,Viajes MBTB,4,"SARIMA(0, 1, 1)×(1, 0, 0, 7)",ok,serie original,558.150,10004.284,10017.669,641,,NaN
7,Viajes MBTB,7,"SARIMA(0, 1, 1)×(0, 0, 0, 0)",ok,serie original,565.877,10051.192,10060.115,641,,NaN
8,Viajes MBTB,8,"SARIMA(1, 1, 1)×(1, 0, 0, 7)",ok,serie original,729.505,9889.125,9906.970,641,,NaN
11,Viajes MBTB,11,"SARIMA(1, 1, 1)×(0, 0, 0, 0)",ok,serie original,730.982,9934.362,9947.747,641,,NaN
2,Viajes MBTB,2,"SARIMA(1, 0, 0)×(1, 0, 1, 7)",ok,serie original,746.095,9784.175,9806.490,641,,NaN
1,Viajes MBTB,1,"SARIMA(1, 0, 0)×(0, 1, 1, 7)",ok,serie original,817.319,9683.597,9696.953,641,,NaN


**Modelo seleccionado:** SARIMA(0, 1, 1)×(0, 1, 1, 7). Parámetros calculados con todos los datos de 2023–2024:

,serie,coeficiente,error_estandar,p_valor,IC95_inferior,IC95_superior
parametro,,,,,,
ma.L1,Viajes MBTB,-0.7957,0.0200,0.0000,-0.8350,-0.7564
ma.S.L7,Viajes MBTB,-0.9998,0.9203,0.2773,-2.8036,0.8040
sigma2,Viajes MBTB,263125.5963,239622.2817,0.2722,-206525.4457,732776.6383


### Temperatura

...

### Precipitación

,serie,id,modelo,estado,familia,MAE_validacion,AIC,BIC,n_ajuste,advertencias,seleccionado
3,Precipitación,3,"SARIMA(1, 0, 1)×(0, 0, 0, 0)",ok,serie original,4.549,4447.744,4465.596,641,,True
1,Precipitación,1,"SARIMA(1, 0, 0)×(0, 0, 0, 0)",ok,serie original,4.561,4451.917,4465.306,641,,NaN
2,Precipitación,2,"SARIMA(0, 0, 1)×(0, 0, 0, 0)",ok,serie original,4.567,4462.273,4475.662,641,,NaN
0,Precipitación,0,"SARIMA(0, 0, 0)×(0, 0, 0, 0)",ok,serie original,4.572,4505.334,4514.260,641,,NaN


**Modelo seleccionado:** SARIMA(1, 0, 1)×(0, 0, 0, 0). Parámetros calculados con todos los datos de 2023–2024:

,serie,coeficiente,error_estandar,p_valor,IC95_inferior,IC95_superior
parametro,,,,,,
intercept,Precipitación,1.3172,0.5994,0.0280,0.1424,2.4920
ar.L1,Precipitación,0.5083,0.1072,0.0000,0.2982,0.7185
ma.L1,Precipitación,-0.2807,0.1169,0.0164,-0.5098,-0.0516
sigma2,Precipitación,66.8610,1.9961,0.0000,62.9488,70.7732


,serie,modelo,n_train,n_estimacion,MAE_validacion,AIC_train,BIC_train,advertencias_train
0,Viajes MBTB,"SARIMA(0, 1, 1)×(0, 1, 1, 7)",731,731,424.572,11109.444,11123.194,
1,Temperatura,"SARIMA(1, 0, 0)×(0, 1, 0, 365)",731,366,3.141,1933.713,1941.518,
2,Humedad,"SARIMA(1, 0, 0)×(0, 0, 0, 0)",731,731,10.388,5136.946,5150.729,
3,Precipitación,"SARIMA(1, 0, 1)×(0, 0, 0, 0)",731,731,4.549,5154.686,5173.064,


## Lectura de los resultados

A continuación se presenta el modelo elegido para cada serie. Los AIC y BIC de la tabla de comparación se calcularon antes de la validación. Los de la tabla resumen se calcularon después, con todo el entrenamiento. Como utilizan distintos datos, no deben compararse entre esas dos tablas.

In [3]:
for fila in resumen.itertuples():
    c = seleccionados[fila.serie]
    lectura = ('Entre las opciones probadas, funcionó mejor restar el valor de hace 365 días.' if c['anual'] else
               'El modelo elegido no necesita restar el valor de hace 365 días.')
    display(Markdown(f'**{fila.serie}:** se seleccionó **{fila.modelo}**, con MAE de validación '
                     f'**{fila.MAE_validacion:.3f}** en las unidades de la serie. {lectura} '
                     'En el punto 6 se revisa cómo predice los datos de 2025.'))


**Viajes MBTB:** se seleccionó **SARIMA(0, 1, 1)×(0, 1, 1, 7)**, con MAE de validación **424.572** en las unidades de la serie. El modelo elegido no necesita restar el valor de hace 365 días. En el punto 6 se revisa cómo predice los datos de 2025.

**Temperatura:** se seleccionó **SARIMA(1, 0, 0)×(0, 1, 0, 365)**, con MAE de validación **3.141** en las unidades de la serie. Entre las opciones probadas, funcionó mejor restar el valor de hace 365 días. En el punto 6 se revisa cómo predice los datos de 2025.

**Humedad:** se seleccionó **SARIMA(1, 0, 0)×(0, 0, 0, 0)**, con MAE de validación **10.388** en las unidades de la serie. El modelo elegido no necesita restar el valor de hace 365 días. En el punto 6 se revisa cómo predice los datos de 2025.

**Precipitación:** se seleccionó **SARIMA(1, 0, 1)×(0, 0, 0, 0)**, con MAE de validación **4.549** en las unidades de la serie. El modelo elegido no necesita restar el valor de hace 365 días. En el punto 6 se revisa cómo predice los datos de 2025.

## Limitaciones y herramientas

Se utiliza `SARIMAX` de statsmodels para calcular los parámetros mediante máxima verosimilitud: un método que busca valores compatibles con los datos observados. Cada serie se modela por separado, sin agregar otras variables explicativas. Pandas y NumPy se usan para organizar los datos.

El método supone errores con distribución normal. Esta idea puede resultar poco adecuada para la precipitación, que tiene muchos días sin lluvia y algunos valores muy altos. Además, estos modelos pueden producir valores negativos, que no tienen sentido para la lluvia o los viajes. Los errores se calculan con las predicciones originales, sin modificarlas.

Solo hay dos años de entrenamiento y se usa un único período de validación. Con otro período, podría elegirse un modelo diferente. En el punto 8 se revisarán los errores del modelo para evaluar si quedaron patrones sin explicar.

**Referencia:** Statsmodels developers. (s. f.). *statsmodels.tsa.statespace.sarimax.SARIMAX*. [Documentación oficial](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.sarimax.SARIMAX.html).